# Full Batch 1 ChunkFormer ingestion

Video ZIPs are read from `MyDrive/AIC2025/video_batch_1`, copied one at a time to Colab SSD, and deleted after transcription. Resumable JSON goes to Drive. ChunkFormer is CC-BY-NC-4.0; confirm this is compatible with competition use.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

!pip -q install chunkformer
!mkdir -p /content/work

In [ ]:
import csv
import json
import re
import shutil
import sqlite3
import subprocess
from pathlib import Path

import torch
from chunkformer import ChunkFormerModel

release_name = "full-v1"
worker_index = 0
worker_count = 1
work = Path("/content/work")
archives_dir = Path("/content/drive/MyDrive/AIC2025/video_batch_1")
output = Path("/content/drive/MyDrive/AIC_ARTIFACTS")
transcripts_dir = output / "workers" / "asr"
release_dir = output / "releases" / release_name
transcripts_dir.mkdir(parents=True, exist_ok=True)
release_dir.mkdir(parents=True, exist_ok=True)

all_archive_paths = sorted(archives_dir.glob("Videos_L*.zip"))
if not all_archive_paths:
    raise ValueError(f"no video ZIPs found in {archives_dir}")
if worker_index < 0 or worker_index >= worker_count:
    raise ValueError("worker_index must be smaller than worker_count")
archive_paths = []
for index, archive_path in enumerate(all_archive_paths):
    if index % worker_count == worker_index:
        archive_paths.append(archive_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = ChunkFormerModel.from_pretrained("khanhld/chunkformer-ctc-large-vie").to(device)
print(f"worker {worker_index + 1}/{worker_count}: {len(archive_paths)} of {len(all_archive_paths)} archives")

In [ ]:
def timestamp_seconds(value):
    hours, minutes, seconds, milliseconds = map(int, value.split(":"))
    return hours * 3600 + minutes * 60 + seconds + milliseconds / 1000


def transcribe(video_path):
    audio_path = Path("/content") / f"{video_path.stem}.wav"
    subprocess.run([
        "ffmpeg", "-y", "-i", str(video_path), "-vn", "-ac", "1", "-ar", "16000", str(audio_path),
    ], check=True)
    decoded = model.endless_decode(
        audio_path=str(audio_path),
        chunk_size=64,
        left_context_size=128,
        right_context_size=128,
        total_batch_duration=14400,
        return_timestamps=True,
    )
    audio_path.unlink()
    segments = []
    for item in decoded:
        segments.append({
            "start": timestamp_seconds(item["start"]),
            "end": timestamp_seconds(item["end"]),
            "text": item["decode"],
        })
    return segments


for source_archive in archive_paths:
    archive_path = work / source_archive.name
    extract_dir = work / source_archive.stem
    shutil.copy2(source_archive, archive_path)
    subprocess.run(["unzip", "-q", str(archive_path), "-d", str(extract_dir)], check=True)
    video_paths = sorted(extract_dir.rglob("L*_V*.mp4"))
    for video_path in video_paths:
        video_id = video_path.stem
        transcript_path = transcripts_dir / f"{video_id}.json"
        if transcript_path.exists():
            print(f"skip {video_id}")
            continue
        segments = transcribe(video_path)
        temporary_path = transcript_path.with_suffix(".json.tmp")
        temporary_path.write_text(json.dumps(segments, ensure_ascii=False), encoding="utf-8")
        temporary_path.replace(transcript_path)
        print(f"saved {video_id} {len(segments)} segments")
    shutil.rmtree(extract_dir)
    archive_path.unlink()

In [ ]:
with (release_dir / "frames.csv").open(newline="", encoding="utf-8") as file:
    expected_videos = {row["video_id"] for row in csv.DictReader(file)}

transcript_paths = sorted(transcripts_dir.glob("L*_V*.json"))
transcript_videos = {path.stem for path in transcript_paths}
if transcript_videos != expected_videos:
    raise ValueError(f"have {len(transcript_videos)} transcripts for {len(expected_videos)} Batch 1 videos")

database_path = release_dir / "asr.sqlite"
temporary_path = release_dir / "asr.sqlite.tmp"
temporary_path.unlink(missing_ok=True)
segments_count = 0
with sqlite3.connect(temporary_path) as connection:
    connection.execute("CREATE VIRTUAL TABLE asr USING fts5(video_id UNINDEXED, start_sec UNINDEXED, end_sec UNINDEXED, text)")
    for transcript_path in transcript_paths:
        video_id = transcript_path.stem
        segments = json.loads(transcript_path.read_text(encoding="utf-8"))
        for segment in segments:
            connection.execute(
                "INSERT INTO asr(video_id, start_sec, end_sec, text) VALUES (?, ?, ?, ?)",
                (video_id, segment["start"], segment["end"], segment["text"]),
            )
            segments_count += 1
temporary_path.replace(database_path)
print(f"published {release_name}/asr.sqlite with {segments_count} segments")
shutil.rmtree(work)

In [ ]:
query = "người nói chuyện"
terms = []
for term in re.findall(r"\w+", query, flags=re.UNICODE):
    terms.append(f'"{term}"')
with sqlite3.connect(database_path) as connection:
    rows = connection.execute(
        "SELECT video_id, start_sec, end_sec, text, bm25(asr) FROM asr WHERE asr MATCH ? ORDER BY bm25(asr) LIMIT 5",
        (" OR ".join(terms),),
    ).fetchall()
for video_id, start, end, text, score in rows:
    print(video_id, start, end, text, score)